# Structural Cell Operations (`window.craftStruct`)

> **Phase 3 structural features:** **Multi-Cell Batch Bar (2.4)**, **Merge Cells (2.2)**, **Partition Dialog — Split / Fork / Extract (2.1)**, **Send Cell(s) to Another Dialog (2.6)**, **Drag + Modifier & Keyboard Cell Reordering (2.3)**, and **Section-Level Actions on Markdown Headings (2.5)**.

## Templates (`#| export_ft`)

In [ ]:
#| export_ft CRAFT.js
#| eval: false
TEMPLATES = {
    'struct_batch_bar': Div(
        Span('0 selected', cls='craft-batch-count', data_ref='count'),
        craft_btn('All-But', size='xs', data_act='allbut'),
        craft_btn('Omit', size='xs', data_act='omit'),
        craft_btn('Hide AI', size='xs', data_act='hide'),
        craft_btn('Partition', size='xs', data_act='partition'),
        craft_btn('Merge', size='xs', data_act='merge', data_ref='mergeBtn'),
        craft_btn('↑', size='xs', data_act='up', aria_label='Move selection up', title='Move up (Alt+Up)'),
        craft_btn('↓', size='xs', data_act='down', aria_label='Move selection down', title='Move down (Alt+Down)'),
        craft_btn('Send to…', size='xs', data_act='send'),
        craft_btn('Delete', size='xs', cls='craft-btn-danger', data_act='delete'),
        id='craft-batch-bar', role='toolbar', aria_label='Multi-cell batch actions',
    ),
    'struct_partition_modal': craft_overlay(
        craft_card(
            Div(
                Strong('Partition Dialog'),
                Button('×', type='button', cls='craft-modal-x', aria_label='Close', data_ref='close'),
                cls='craft-modal-head',
            ),
            Div(
                Div(
                    'Note: Each Solveit dialog owns its own kernel (kid). The newly created dialog will start with a fresh kernel state.',
                    cls='craft-ctx-prev-note',
                ),
                Div(
                    Label(Input(type='radio', name='craft-part-mode', value='split', checked=True), Span(' Split here — keep cells above, move cut & below to new dialog'), cls='craft-part-opt'),
                    Label(Input(type='radio', name='craft-part-mode', value='fork'), Span(' Fork from here — duplicate dialog and drop everything below cut'), cls='craft-part-opt'),
                    Label(Input(type='radio', name='craft-part-mode', value='extract'), Span(' Extract selection — copy or move selected cells to new dialog'), cls='craft-part-opt'),
                    cls='craft-part-modes', data_ref='modes',
                ),
                Div(
                    Label('New dialog name', cls='craft-part-lbl'),
                    craft_input(placeholder='new_dialog.ipynb', data_ref='name'),
                    cls='craft-part-field',
                ),
                Div(
                    Label(Input(type='checkbox', data_ref='moveSel'), Span(' Remove extracted cells from current dialog (move)'), cls='craft-part-opt'),
                    Label(Input(type='checkbox', data_ref='leaveLink', checked=True), Span(' Leave a note with a link to the new dialog'), cls='craft-part-opt'),
                    cls='craft-part-extra', data_ref='extractOpts',
                ),
                cls='craft-part-body',
            ),
            Div(
                craft_btn('Cancel', size='xs', data_ref='cancel'),
                craft_btn('Create dialog', variant='primary', size='xs', data_ref='go'),
                cls='craft-modal-foot',
            ),
            cls='craft-part-card',
        ),
        id='craft-partition-modal', role='dialog', aria_modal='true', aria_label='Partition Dialog',
    ),
    'struct_send_modal': craft_overlay(
        craft_card(
            Div(
                Strong('Send Cell(s) to Another Dialog'),
                Button('×', type='button', cls='craft-modal-x', aria_label='Close', data_ref='close'),
                cls='craft-modal-head',
            ),
            Div(
                craft_input(placeholder='Filter dialogs…', data_ref='filter'),
                Label(Input(type='checkbox', data_ref='moveFlag'), Span(' Move (delete from current dialog after copying)'), cls='craft-part-opt'),
                Div(cls='craft-send-list', data_ref='list'),
                cls='craft-part-body',
            ),
            Div(
                craft_btn('Cancel', size='xs', data_ref='cancel'),
                cls='craft-modal-foot',
            ),
            cls='craft-part-card',
        ),
        id='craft-send-dialog-modal', role='dialog', aria_modal='true', aria_label='Send cells to another dialog',
    ),
    'struct_sec_menu': craft_menu(id='craft-sec-menu', role='menu'),
}

## Cell Source Helpers, Merge & Notebook I/O (`#| export_js_iife`)


In [ ]:
#| export_js_iife CRAFT.js craftStruct
#| eval: false
window.craftStruct?.disable?.();
if (!document.getElementById('dialog-container')) return;

const CORE = window.__craftCMCore;
if (!CORE) return;
const {
  el, qsa, on, store, mountTpl, mountModal, htmxPost, onDomChange,
  notify, notifyErr, normId, getCellEl, typingBlocked,
  allCells, selectedCells, primaryCell, cellMtype, deleteCellEl,
} = CORE;
const ac = new AbortController(), sig = {signal: ac.signal}, cap = {capture: true, signal: ac.signal};
const DRAG_MOD_KEY = 'craft-reorder-mod'; // 'alt' (default) | 'hyper'

async function commitEditors() {
  if (window.craftC2?.active?.()) await window.craftC2.exit();
  if (window.craftMd?.active?.()) await window.craftMd.exit();
}

function getCellInputText(cell) {
  if (!cell) return null;
  const id = normId(cell.id);
  const inp = cell.querySelector(`textarea[name="content"], input[name="content"], textarea#_${id}_content, textarea[data-content]`);
  if (inp && typeof inp.value === 'string') return inp.value;
  const mtype = cellMtype(cell);
  if (mtype === 'code' || mtype === 'raw') {
    const pre = cell.querySelector(':scope > .card-in .msg-content pre code, :scope > .card-in .msg-content pre');
    if (pre) return pre.textContent || '';
  }
  return null;
}

/* ── 2.2 Merge Cells ───────────────────────────────────────────────── */
function canMerge(cells = selectedCells()) {
  if (cells.length < 2) return false;
  const all = allCells();
  const idxs = cells.map(c => all.indexOf(c));
  if (idxs.some(i => i < 0)) return false;
  for (let i = 1; i < idxs.length; i++) if (idxs[i] !== idxs[i - 1] + 1) return false;
  const t0 = cellMtype(cells[0]);
  return cells.every(c => cellMtype(c) === t0);
}

async function loadDialogRawSources() {
  const nb = await fetchDialogNb(currentDlgPath());
  const map = new Map();
  for (const c of nb?.cells || []) {
    const cid = normId(c?.id || '');
    if (!cid) continue;
    map.set(cid, Array.isArray(c.source) ? c.source.join('') : String(c.source ?? ''));
  }
  return map;
}

async function mergeSelectedCells(cells = selectedCells()) {
  if (!canMerge(cells)) {
    notifyErr('Select 2 or more consecutive cells of the same type to merge.', 'craftStruct');
    return false;
  }
  await commitEditors();
  let rawMap = null;
  const parts = [];
  for (const c of cells) {
    let txt = getCellInputText(c);
    if (txt === null) {
      if (!rawMap) rawMap = await loadDialogRawSources();
      const id = normId(c.id);
      if (rawMap.has(id)) txt = rawMap.get(id);
    }
    if (txt === null) {
      notifyErr('Could not read raw source for one or more cells; save or open them before merging.', 'craftStruct');
      return false;
    }
    parts.push(txt);
  }
  const combined = parts.join('\n\n');
  const firstId = normId(cells[0].id);
  try {
    await htmxPost('/set_contents_', {id_: '_' + firstId, ids: firstId, contents: JSON.stringify({[firstId]: combined})});
    for (let i = cells.length - 1; i >= 1; i--) await deleteCellEl(cells[i], {tag: 'craftStruct'});
    notify(`Merged ${cells.length} cells.`, 'success');
    return true;
  } catch (err) {
    notifyErr('Could not merge cells.', 'craftStruct');
    console.error('craftStruct: merge failed', err);
    return false;
  }
}

/* ── .ipynb helpers for Partition (2.1) & Send-to-Dialog (2.6) ─────── */
function currentDlgPath() {
  const ctx = window.solveitFileTree?.getCurrentContext?.();
  if (ctx?.dlgPath) return ctx.dlgPath;
  const u = new URL(location.href);
  const name = u.searchParams.get('name');
  return name ? (name.endsWith('.ipynb') ? name : `${name}.ipynb`) : null;
}

function parentDirOf(p) {
  const parts = String(p || '').replace(/^\/+/, '').split('/').filter(Boolean);
  parts.pop();
  return parts.length ? parts.join('/') : '.';
}

function staticUrl(relPath) {
  return '/static/' + String(relPath || '').replace(/^\/+/, '').split('/').map(encodeURIComponent).join('/');
}

function domCellToNbCell(cell) {
  const mtype = cellMtype(cell);
  const src = getCellInputText(cell) ?? (cell.querySelector(':scope > .card-in .msg-content')?.textContent || '').trim();
  return {
    cell_type: mtype === 'code' ? 'code' : (mtype === 'raw' ? 'raw' : 'markdown'),
    id: normId(cell.id),
    metadata: {solveit_msg_type: mtype},
    source: src.split(/(?<=\n)/),
    ...(mtype === 'code' ? {execution_count: null, outputs: []} : {}),
  };
}

async function fetchDialogNb(relPath) {
  if (!relPath) return null;
  try {
    const r = await fetch(staticUrl(relPath), {credentials: 'same-origin'});
    if (!r.ok) return null;
    return await r.json();
  } catch { return null; }
}

async function uploadDialogNb(folder, filename, nbObj) {
  const fname = filename.endsWith('.ipynb') ? filename : `${filename}.ipynb`;
  const fd = new FormData();
  const blob = new Blob([JSON.stringify(nbObj, null, 1)], {type: 'application/x-ipynb+json'});
  fd.append('files', new File([blob], fname, {type: 'application/x-ipynb+json'}));
  const r = await fetch(`/upload_file_?folder=${encodeURIComponent(folder || '.')}`, {
    method: 'POST', credentials: 'same-origin', headers: {'HX-Request': 'true'}, body: fd,
  });
  if (!r.ok) throw new Error(`HTTP ${r.status}`);
  window.solveitFileTree?.refresh?.();
  return folder && folder !== '.' ? `${folder}/${fname}` : fname;
}


## Partition Dialog & Send-to-Dialog Modals (`#| export_js_iife`)

In [ ]:
#| export_js_iife CRAFT.js craftStruct
#| eval: false
/* ── 2.1 Partition Dialog Modal ────────────────────────────────────── */
let partModal = null, partTargetCells = [];

async function executePartition({mode, name, moveSel, leaveLink}) {
  const cleanName = String(name || '').trim().replace(/\.ipynb$/i, '');
  if (!cleanName) return notifyErr('Enter a name for the new dialog.', 'craftStruct');
  await commitEditors();
  const dlgPath = currentDlgPath();
  const folder = parentDirOf(dlgPath);
  const baseNb = (await fetchDialogNb(dlgPath)) || {
    cells: allCells().map(domCellToNbCell),
    metadata: {}, nbformat: 4, nbformat_minor: 5,
  };
  const all = allCells();
  const cutCell = partTargetCells[0] || primaryCell() || all[0];
  const cutIdx = Math.max(0, all.indexOf(cutCell));
  const findNbCells = domList => {
    const ids = new Set(domList.map(c => normId(c.id)));
    const matched = (baseNb.cells || []).filter(c => ids.has(normId(c.id)));
    return matched.length ? matched : domList.map(domCellToNbCell);
  };

  let newCells = [], toDelete = [];
  if (mode === 'split') {
    const below = all.slice(cutIdx);
    newCells = findNbCells(below);
    toDelete = below;
  } else if (mode === 'fork') {
    const above = all.slice(0, cutIdx + 1);
    newCells = findNbCells(above);
  } else {
    const chosen = partTargetCells.length ? partTargetCells : [cutCell];
    newCells = findNbCells(chosen);
    if (moveSel) toDelete = chosen;
  }

  try {
    const newPath = await uploadDialogNb(folder, cleanName, {...baseNb, cells: newCells});
    if (toDelete.length) {
      for (let i = toDelete.length - 1; i >= 0; i--) await deleteCellEl(toDelete[i], {tag: 'craftStruct'});
    }
    if (mode === 'extract' && leaveLink) {
      const linkText = `> Extracted ${newCells.length} cell(s) to [${cleanName}](/dialog_?name=${encodeURIComponent(newPath.replace(/\.ipynb$/, ''))})`;
      await htmxPost('/upsert_msg_', {id_: '', msg_type: 'note', is_input: '1', content: linkText});
    }
    notify(`Created dialog "${cleanName}" (${newCells.length} cells).`, 'success');
    return newPath;
  } catch (err) {
    notifyErr(`Partition failed: ${err.message || err}`, 'craftStruct');
    console.error('craftStruct: partition failed', err);
    return null;
  }
}

function openPartitionModal(cells = null) {
  partTargetCells = cells?.length ? cells : (selectedCells().length ? selectedCells() : [primaryCell()].filter(Boolean));
  if (!partTargetCells.length) return notifyErr('Select a cell first.', 'craftStruct');
  if (!partModal) {
    partModal = mountModal('struct_partition_modal', {refs: true, signal: ac.signal});
    on(partModal.cancel, 'click', () => partModal.close(), sig);
    const syncExtract = () => {
      const m = partModal.modes.querySelector('input:checked')?.value;
      partModal.extractOpts.hidden = m !== 'extract';
    };
    on(partModal.modes, 'change', syncExtract, sig);
    on(partModal.go, 'click', async () => {
      const mode = partModal.modes.querySelector('input:checked')?.value || 'split';
      const res = await executePartition({
        mode, name: partModal.name.value,
        moveSel: partModal.moveSel.checked, leaveLink: partModal.leaveLink.checked,
      });
      if (res) partModal.close();
    }, sig);
    syncExtract();
  }
  const base = (currentDlgPath() || 'dialog').split('/').pop().replace(/\.ipynb$/, '');
  partModal.name.value = `${base}_part`;
  if (partTargetCells.length > 1) {
    const rad = partModal.modes.querySelector('input[value="extract"]');
    if (rad) { rad.checked = true; partModal.extractOpts.hidden = false; }
  }
  partModal.open();
  partModal.name.focus(); partModal.name.select();
}

/* ── 2.6 Send Cell(s) to Another Dialog ────────────────────────────── */
let sendModal = null, sendCells = [], sendEntries = [];

async function sendCellsToTarget(targetNode, move) {
  await commitEditors();
  const srcPath = currentDlgPath();
  const srcNb = await fetchDialogNb(srcPath);
  const targetPath = targetNode.path.endsWith('.ipynb') ? targetNode.path : `${targetNode.path}.ipynb`;
  const targetNb = (await fetchDialogNb(targetPath)) || {cells: [], metadata: {}, nbformat: 4, nbformat_minor: 5};
  const ids = new Set(sendCells.map(c => normId(c.id)));
  const nbCells = srcNb?.cells?.filter(c => ids.has(normId(c.id))) || [];
  const toAppend = nbCells.length ? nbCells : sendCells.map(domCellToNbCell);
  targetNb.cells = [...(targetNb.cells || []), ...toAppend];
  try {
    await uploadDialogNb(parentDirOf(targetPath), targetPath.split('/').pop(), targetNb);
    if (move) {
      for (let i = sendCells.length - 1; i >= 0; i--) await deleteCellEl(sendCells[i], {tag: 'craftStruct'});
    }
    notify(`${move ? 'Moved' : 'Copied'} ${toAppend.length} cell(s) to ${targetNode.name}.`, 'success');
  } catch (err) {
    notifyErr(`Could not send cells: ${err.message || err}`, 'craftStruct');
  }
}

function renderSendList() {
  if (!sendModal) return;
  const q = (sendModal.filter.value || '').trim().toLowerCase();
  const cur = currentDlgPath();
  const matches = sendEntries.filter(n => n.kind === 'dialog' && n.path !== cur && (!q || n.name.toLowerCase().includes(q) || n.path.toLowerCase().includes(q)));
  sendModal.list.replaceChildren(...(matches.length ? matches.slice(0, 50).map(n => el('button', {
    type: 'button', cls: 'craft-send-row',
    onClick: async () => {
      sendModal.close();
      await sendCellsToTarget(n, sendModal.moveFlag.checked);
    },
  }, el('strong', {}, n.name), el('span', {cls: 'craft-send-path'}, n.path))) : [el('div', {cls: 'craft-empty-state'}, 'No matching dialogs loaded.')]));
}

async function openSendModal(cells = null) {
  sendCells = cells?.length ? cells : (selectedCells().length ? selectedCells() : [primaryCell()].filter(Boolean));
  if (!sendCells.length) return notifyErr('Select a cell first.', 'craftStruct');
  if (!sendModal) {
    sendModal = mountModal('struct_send_modal', {refs: true, signal: ac.signal});
    on(sendModal.cancel, 'click', () => sendModal.close(), sig);
    on(sendModal.filter, 'input', renderSendList, sig);
  }
  sendModal.filter.value = '';
  sendModal.open();
  sendEntries = (await window.solveitFileTree?.entries?.()) || [];
  renderSendList();
  sendModal.filter.focus();
}

## Keyboard & Drag-Modifier Cell Reordering (`#| export_js_iife`)

In [ ]:
#| export_js_iife CRAFT.js craftStruct
#| eval: false
/* ── 2.3 Keyboard & Drag-Modifier Cell Reordering ──────────────────── */
const fireKey = (key, target = document.body) => {
  target.dispatchEvent(new KeyboardEvent('keydown', {key, bubbles: true, cancelable: true}));
};

function waitForDomSettle(pred, timeout = 2000) {
  return new Promise(resolve => {
    if (pred()) return resolve(true);
    let settled = false;
    const onAbort = () => done(false);
    const done = ok => {
      if (settled) return;
      settled = true;
      clearTimeout(t);
      off?.();
      ac.signal.removeEventListener('abort', onAbort);
      resolve(ok);
    };
    const off = onDomChange(() => { if (pred()) done(true); }, {delay: 30, inDialog: true});
    const t = setTimeout(() => done(!!pred()), timeout);
    ac.signal.addEventListener('abort', onAbort, {once: true});
  });
}

async function cutAndPasteAfter(block, afterId) {
  const all = allCells();
  for (const c of all) c.setAttribute('data-sm', 'unselected');
  block.forEach((c, i) => c.setAttribute('data-sm', i === 0 ? 'primary' : 'secondary'));
  const cutIds = block.map(c => normId(c.id));
  fireKey('x');
  const cutOk = await waitForDomSettle(() => cutIds.every(id => !getCellEl(id)), 2500);
  if (!cutOk) return false;
  const target = getCellEl(afterId);
  if (!target) return false;
  window.selectMsg?.(window.$?.(target));
  await waitForDomSettle(() => target.getAttribute('data-sm') === 'primary', 400);
  fireKey('v');
  return await waitForDomSettle(() => cutIds.every(id => !!getCellEl(id)), 2500);
}

async function moveCells(cells, delta) {
  const all = allCells();
  const ordered = cells.filter(c => all.includes(c)).sort((a, b) => all.indexOf(a) - all.indexOf(b));
  if (!ordered.length || !delta) return false;
  const firstIdx = all.indexOf(ordered[0]), lastIdx = all.indexOf(ordered.at(-1));
  const targetIdx = delta < 0 ? firstIdx + delta : lastIdx + delta;
  if (targetIdx < 0 || targetIdx >= all.length) return false;
  await commitEditors();

  try {
    if (delta < 0 && targetIdx === 0) {
      const firstId = normId(all[0].id);
      const lastBlockId = normId(ordered.at(-1).id);
      if (firstIdx === 1) {
        return await cutAndPasteAfter([all[0]], lastBlockId);
      }
      if (!await cutAndPasteAfter(ordered, firstId)) return false;
      const liveFirst = getCellEl(firstId);
      if (!liveFirst) return false;
      return await cutAndPasteAfter([liveFirst], lastBlockId);
    }
    const anchor = all[targetIdx];
    if (!anchor) return false;
    const afterCell = delta < 0 ? all[targetIdx - 1] : anchor;
    if (!afterCell) return false;
    return await cutAndPasteAfter(ordered, normId(afterCell.id));
  } catch (err) {
    console.error('craftStruct: reorder failed', err);
    return false;
  }
}

let dragState = null;
const dropLine = el('div', {cls: 'craft-drop-line', hidden: true});
document.body.append(dropLine);

const hasDragMod = e => {
  const mode = store.get(DRAG_MOD_KEY, 'alt');
  return mode === 'hyper' ? (e.ctrlKey && e.altKey && e.shiftKey && e.metaKey) : (e.altKey && !e.ctrlKey && !e.metaKey);
};

on(document, 'pointerdown', e => {
  if (window.craftMobile?.active?.() || e.button !== 0 || !hasDragMod(e) || typingBlocked()) return;
  const cell = e.target.closest?.('#dialog-container > .editable[id]');
  if (!cell) return;
  e.preventDefault();
  dragState = {cell, startY: e.clientY, active: false, targetCell: null, before: true};
}, cap);

on(document, 'pointermove', e => {
  if (!dragState) return;
  if (!dragState.active && Math.abs(e.clientY - dragState.startY) > 6) {
    dragState.active = true;
    dragState.cell.classList.add('craft-dragging');
  }
  if (!dragState.active) return;
  const cells = allCells().filter(c => c !== dragState.cell && c.offsetParent !== null);
  let best = null, before = true, bestDist = Infinity;
  for (const c of cells) {
    const r = c.getBoundingClientRect();
    const dTop = Math.abs(e.clientY - r.top), dBot = Math.abs(e.clientY - r.bottom);
    if (dTop < bestDist) { bestDist = dTop; best = c; before = true; }
    if (dBot < bestDist) { bestDist = dBot; best = c; before = false; }
  }
  if (best) {
    dragState.targetCell = best; dragState.before = before;
    const r = best.getBoundingClientRect();
    dropLine.hidden = false;
    dropLine.style.left = r.left + 'px';
    dropLine.style.width = r.width + 'px';
    dropLine.style.top = (before ? r.top : r.bottom) + 'px';
  }
}, sig);

on(document, 'pointerup', async () => {
  if (!dragState) return;
  const st = dragState;
  dragState = null;
  st.cell.classList.remove('craft-dragging');
  dropLine.hidden = true;
  if (!st.active || !st.targetCell) return;
  const all = allCells();
  const fromIdx = all.indexOf(st.cell), toIdx = all.indexOf(st.targetCell);
  if (fromIdx < 0 || toIdx < 0 || fromIdx === toIdx) return;
  const delta = (st.before ? toIdx : toIdx + 1) - fromIdx;
  const adj = delta > 0 ? delta - 1 : delta;
  if (adj !== 0) await moveCells([st.cell], adj);
}, sig);

## Batch Bar, Heading Section Menu, Hotkeys & Public API (`#| export_js_iife`)

In [ ]:
#| export_js_iife CRAFT.js craftStruct
#| eval: false
/* ── 2.4 Contextual Multi-Cell Batch Bar ───────────────────────────── */
let batchBar = null;

function syncBatchBar() {
  const sel = selectedCells();
  const show = sel.length >= 2 && !typingBlocked() && !document.documentElement.classList.contains('craft-ab-staging');
  if (!show) { batchBar?.root?.remove(); return; }
  if (!batchBar) {
    batchBar = mountTpl('struct_batch_bar', {refs: true});
    on(batchBar.root, 'click', async e => {
      const btn = e.target.closest('button[data-act]');
      if (!btn) return;
      e.preventDefault(); e.stopPropagation();
      const cur = selectedCells();
      const act = btn.dataset.act;
      if (act === 'allbut') window.craftContext?.stageScope?.(cur.map(c => c.id), 'All-But');
      else if (act === 'omit') window.craftOmit?.toggle?.();
      else if (act === 'hide') {
        const anyVisible = cur.some(c => !window.craftContext?.isHiddenFromAI?.(c));
        for (const c of cur) await window.craftContext?.setHiddenFromAI?.(c, anyVisible);
        window.craftContext?.updateMeter?.();
      } else if (act === 'partition') openPartitionModal(cur);
      else if (act === 'merge') await mergeSelectedCells(cur);
      else if (act === 'up') await moveCells(cur, -1);
      else if (act === 'down') await moveCells(cur, 1);
      else if (act === 'send') openSendModal(cur);
      else if (act === 'delete') {
        if (confirm(`Delete ${cur.length} selected cells?`)) {
          for (let i = cur.length - 1; i >= 0; i--) await deleteCellEl(cur[i], {tag: 'craftStruct'});
        }
      }
    }, sig);
  }
  if (!batchBar.root.isConnected) document.body.append(batchBar.root);
  batchBar.count.textContent = `${sel.length} selected`;
  if (batchBar.mergeBtn) batchBar.mergeBtn.disabled = !canMerge(sel);
}

/* ── 2.5 Section-Level Actions on Markdown Headings ────────────────── */
const secMenu = mountTpl('struct_sec_menu');
document.body.append(secMenu);
let secMenuCell = null;

const closeSecMenu = () => { secMenu.classList.remove('open'); secMenuCell = null; };

const getSectionFor = cell => window.craftContext?.getSectionCells?.(cell) || [cell];

function openSecMenu(trigger, cell) {
  if (secMenuCell === cell && secMenu.classList.contains('open')) return closeSecMenu();
  secMenuCell = cell;
  const sec = getSectionFor(cell);
  const codeCells = sec.filter(c => cellMtype(c) === 'code');
  const items = [
    {label: `Run all code in section (${codeCells.length})`, disabled: !codeCells.length, fn: async () => {
      for (const c of codeCells) await CORE.runCellCmd({id: normId(c.id), cmd: 'run', tag: 'craftStruct'});
    }},
    {label: `All-But this section (${sec.length})`, fn: () => window.craftContext?.lensHeadingSection?.(cell)},
    {label: 'Omit section', fn: () => {
      for (const c of allCells()) c.setAttribute('data-sm', 'unselected');
      sec.forEach((c, i) => c.setAttribute('data-sm', i === 0 ? 'primary' : 'secondary'));
      window.craftOmit?.toggle?.();
    }},
    {label: 'Partition section to new dialog…', fn: () => openPartitionModal(sec)},
    {label: 'Copy section as Markdown', fn: async () => {
      const md = sec.map(c => {
        const t = cellMtype(c), txt = getCellInputText(c);
        return t === 'code' ? '```python\n' + txt + '\n```' : txt;
      }).join('\n\n');
      try { await navigator.clipboard.writeText(md); notify('Copied section as Markdown.', 'success'); }
      catch { prompt('Section Markdown:', md); }
    }},
  ];
  secMenu.replaceChildren(...items.map(it => el('button', {
    type: 'button', role: 'menuitem', disabled: !!it.disabled,
    onClick: e => { e.stopPropagation(); closeSecMenu(); it.fn(); },
  }, it.label)));
  const r = trigger.getBoundingClientRect();
  secMenu.classList.add('open');
  secMenu.style.left = Math.max(8, Math.min(r.left, innerWidth - secMenu.offsetWidth - 8)) + 'px';
  secMenu.style.top = Math.max(8, Math.min(r.bottom + 4, innerHeight - secMenu.offsetHeight - 8)) + 'px';
}

function syncHeadingTriggers() {
  for (const cell of allCells()) {
    const isHdr = cell.hasAttribute('data-sm-header') || !!cell.querySelector(':scope > .card-in[data-mtype="note"] .msg-content :is(h1,h2,h3,h4,h5,h6)');
    const existing = cell.querySelector(':scope > .craft-sec-trigger');
    if (!isHdr) { existing?.remove(); continue; }
    if (existing) continue;
    const btn = el('button', {
      type: 'button', cls: 'craft-sec-trigger',
      'aria-label': 'Section actions', title: 'Section actions',
      onClick: e => { e.preventDefault(); e.stopPropagation(); openSecMenu(btn, cell); },
    }, '§ ▾');
    cell.append(btn);
  }
}

on(document, 'pointerdown', e => {
  if (secMenu.classList.contains('open') && !secMenu.contains(e.target) && !e.target.closest?.('.craft-sec-trigger')) closeSecMenu();
}, cap);

/* ── Action Menu & Hotkeys ─────────────────────────────────────────── */
const offMenu = window.craftActionMenus?.registerProvider?.(ctx => {
  if (ctx.isOutput || !ctx.cell) return [];
  const items = [
    {header: 'Structure'},
    {label: 'Partition dialog from here…', onClick: () => openPartitionModal([ctx.cell])},
    {label: 'Send cell to another dialog…', onClick: () => openSendModal([ctx.cell])},
    {label: 'Move cell up', onClick: () => moveCells([ctx.cell], -1)},
    {label: 'Move cell down', onClick: () => moveCells([ctx.cell], 1)},
  ];
  if (canMerge()) items.splice(1, 0, {label: 'Merge selected cells', onClick: () => mergeSelectedCells()});
  return items;
});

const reg = spec => window.__hotkeyRegister?.(spec);
reg({id: 'cell-merge', label: 'Merge selected consecutive cells', group: 'CRAFT · Structure', combo: '', handler: () => { mergeSelectedCells(); return true; }});
reg({id: 'cell-partition', label: 'Partition dialog (Split / Fork / Extract)', group: 'CRAFT · Structure', combo: '', handler: () => { openPartitionModal(); return true; }});
reg({id: 'cell-send-dialog', label: 'Send cell(s) to another dialog', group: 'CRAFT · Structure', combo: '', handler: () => { openSendModal(); return true; }});
reg({id: 'cell-move-up', label: 'Move cell(s) up', group: 'CRAFT · Structure', combo: 'Alt+ArrowUp', handler: () => { if (typingBlocked()) return false; const s = selectedCells(); if (!s.length) return false; moveCells(s, -1); return true; }});
reg({id: 'cell-move-down', label: 'Move cell(s) down', group: 'CRAFT · Structure', combo: 'Alt+ArrowDown', handler: () => { if (typingBlocked()) return false; const s = selectedCells(); if (!s.length) return false; moveCells(s, 1); return true; }});
reg({id: 'cell-drag-mod', label: 'Toggle drag-reorder modifier (Alt vs Hyper)', group: 'CRAFT · Structure', combo: '', handler: () => {
  const next = store.get(DRAG_MOD_KEY, 'alt') === 'alt' ? 'hyper' : 'alt';
  store.set(DRAG_MOD_KEY, next);
  notify(`Drag-reorder modifier set to ${next === 'hyper' ? 'Hyper (Ctrl+Alt+Shift+Cmd)' : 'Alt'}.`);
  return true;
}});

const offDom = onDomChange(() => { syncBatchBar(); syncHeadingTriggers(); }, {attrs: ['data-sm', 'class', 'data-sm-header'], delay: 'raf', inDialog: true});
on(document, 'focusin focusout', () => setTimeout(syncBatchBar, 30), sig);

syncBatchBar();
syncHeadingTriggers();

window.craftStruct = {
  canMerge, mergeSelectedCells, openPartitionModal, openSendModal, moveCells, syncBatchBar,
  disable() {
    ac.abort(); offDom?.(); offMenu?.();
    batchBar?.root?.remove(); partModal?.destroy?.(); sendModal?.destroy?.();
    secMenu.remove(); dropLine.remove();
    qsa('.craft-sec-trigger').forEach(b => b.remove());
    delete window.craftStruct;
  },
};

## Structural Operations CSS (`#| export_css`)

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ── Structural Ops: Batch Bar, Partition, Send, Reorder, Section Menu (15_structural_ops) ── */
#craft-batch-bar {
  position: fixed;
  left: 50%;
  bottom: calc(16px + env(safe-area-inset-bottom, 0px));
  transform: translateX(-50%);
  z-index: 1210;
  display: flex;
  flex-wrap: wrap;
  align-items: center;
  gap: 5px;
  max-width: min(95vw, 760px);
  padding: 6px 10px;
  border-radius: 999px;
  background: var(--craft-recessed);
  border: 1px solid var(--craft-accent-border);
  box-shadow: 0 12px 32px rgba(0,0,0,.55);
  font-size: 11px;
}
html.craft-mobile #craft-batch-bar {
  bottom: calc(64px + env(safe-area-inset-bottom, 0px));
  border-radius: 12px;
}
.craft-batch-count { font-weight: 600; color: var(--craft-accent); padding: 0 4px; }

.craft-part-card { width: min(520px, 92vw); display: flex; flex-direction: column; overflow: hidden; }
.craft-part-body { padding: 14px 16px; display: flex; flex-direction: column; gap: 10px; font-size: 12px; }
.craft-part-modes, .craft-part-extra { display: flex; flex-direction: column; gap: 6px; }
.craft-part-opt { display: flex; align-items: center; gap: 6px; cursor: pointer; }
.craft-part-field { display: flex; flex-direction: column; gap: 4px; }
.craft-part-lbl { font-size: 11px; color: var(--craft-muted); }

.craft-send-list { max-height: 260px; overflow-y: auto; display: flex; flex-direction: column; gap: 3px; }
.craft-send-row {
  display: flex; justify-content: space-between; align-items: center; gap: 10px;
  padding: 6px 9px; border-radius: 5px; border: 1px solid var(--craft-border-soft);
  background: var(--craft-bg); color: var(--craft-fg); text-align: left; cursor: pointer; font-size: 11.5px;
}
.craft-send-row:hover { border-color: var(--craft-accent); background: var(--craft-hover); }
.craft-send-path { font-family: var(--craft-code-family); font-size: 10px; color: var(--craft-muted); }

.editable.craft-dragging { opacity: 0.45 !important; }
.craft-drop-line {
  position: fixed; height: 2px; z-index: 1300; pointer-events: none;
  background: var(--craft-accent); box-shadow: 0 0 8px var(--craft-accent);
}

/* Section heading menu trigger */
#dialog-container > .editable:has(> .craft-sec-trigger) { position: relative; }
.craft-sec-trigger {
  position: absolute; top: 2px; right: 34px; z-index: 6;
  padding: 1px 6px; border-radius: 4px; border: 1px solid var(--craft-border);
  background: var(--craft-surface); color: var(--craft-muted); font-size: 10px;
  cursor: pointer; opacity: 0; transition: opacity .12s ease;
}
#dialog-container > .editable:is(:hover, :focus-within) > .craft-sec-trigger,
.craft-sec-trigger:focus-visible,
html.craft-mobile .craft-sec-trigger { opacity: 1; }
.craft-sec-trigger:hover { color: var(--craft-accent); border-color: var(--craft-accent); }
#craft-sec-menu { display: none; }
#craft-sec-menu.open { display: block; }

Verify `15_structural_ops.ipynb` exports `window.craftStruct`, its templates, and CSS.

In [ ]:
import os, subprocess
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/15_structural_ops.ipynb')
js = '\n'.join(code for _, code in assets['CRAFT.js'])
css = '\n'.join(code for _, code in assets['CRAFT.css'])
for sym in ('window.craftStruct', 'struct_batch_bar', 'struct_partition_modal', 'struct_send_modal', 'mergeSelectedCells', 'openPartitionModal'):
    assert sym in js, f"Missing {sym}"
assert '#craft-batch-bar' in css and '.craft-drop-line' in css and '.craft-sec-trigger' in css

node_script = r'''
const assert = require('assert');
const src = process.env.STRUCT_JS;
const i0 = src.indexOf('function canMerge');
const i1 = src.indexOf('async function loadDialogRawSources');
const cells = [{id: 'a', t: 'code'}, {id: 'b', t: 'code'}, {id: 'c', t: 'note'}, {id: 'd', t: 'code'}];
const canMerge = new Function('allCells', 'selectedCells', 'cellMtype',
  src.slice(i0, i1) + '\nreturn canMerge;'
)(() => cells, () => [], c => c.t);

assert.strictEqual(canMerge([]), false);
assert.strictEqual(canMerge([cells[0]]), false);
assert.strictEqual(canMerge([cells[0], cells[1]]), true);
assert.strictEqual(canMerge([cells[1], cells[0]]), false);
assert.strictEqual(canMerge([cells[1], cells[2]]), false);
assert.strictEqual(canMerge([cells[0], cells[3]]), false);
assert.strictEqual(canMerge([cells[0], {id: 'x', t: 'code'}]), false);
'''
subprocess.run(['node', '-e', node_script], env={**os.environ, 'STRUCT_JS': js}, check=True)
